# Day 19 — The `Agent` class

You've written the same ReAct loop a few times now. Time to package it into a reusable
**`Agent`** class so you stop copy-pasting: it owns its tools, builds its own system
prompt, and exposes a single `run(goal)` method. This is the shape every framework gives
you — now you know what's inside.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run the cell. Stuck? The solution is below — but try first.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator, clock, word_count

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

class Agent:
    def __init__(self, tools, system=None, max_steps=6):
        self.tools = tools
        self.max_steps = max_steps
        names = "|".join(list(tools) + ["finish"])
        self.system = system or (
            "Reason then act. Reply ONLY with JSON: "
            '{"thought":"...","action":"' + names + '","action_input":"..."}. '
            "Each tool takes a single string. Use 'finish' with your final answer."
        )

    def run(self, goal, verbose=False):
        # TODO: implement the loop —
        #   seed messages with system + user(goal); each step chat()->parse_json;
        #   if action == "finish" return action_input; else run self.tools[action](action_input)
        #   and append the assistant turn + "Observation: ..." user turn.
        raise NotImplementedError

# agent = Agent({"calculator": calculator, "clock": clock, "word_count": word_count})
# print(agent.run("How many words are in 'the quick brown fox'?"))

## 🔒 Solution

One correct way to do it. Compare it with your version.

In [ ]:
import json
from shared.llm import chat
from shared.tools import calculator, clock, word_count

def parse_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

class Agent:
    def __init__(self, tools, system=None, max_steps=6):
        self.tools = tools
        self.max_steps = max_steps
        names = "|".join(list(tools) + ["finish"])
        self.system = system or (
            "Reason then act. Reply ONLY with JSON: "
            '{"thought":"...","action":"' + names + '","action_input":"..."}. '
            "Each tool takes a single string. Use 'finish' with your final answer."
        )

    def run(self, goal, verbose=False):
        messages = [{"role": "system", "content": self.system},
                    {"role": "user", "content": goal}]
        for step in range(1, self.max_steps + 1):
            obj = parse_json(chat(messages, temperature=0))
            if verbose:
                print(f"[{step}] 💭 {obj.get('thought', '')}")
            if obj.get("action") == "finish":
                return obj.get("action_input", "")
            tool = self.tools.get(obj.get("action"))
            obs = tool(obj.get("action_input", "")) if tool else f"unknown tool: {obj.get('action')}"
            messages.append({"role": "assistant", "content": json.dumps(obj)})
            messages.append({"role": "user", "content": f"Observation: {obs}"})
        return "stopped (hit max_steps)"

agent = Agent({"calculator": calculator, "clock": clock, "word_count": word_count})
print(agent.run("How many words are in 'the quick brown fox'?", verbose=True))